# 09a — Create Governed Tags, Discover Domains & Subdomains

Creates the **governed tags** and **Discover domains/subdomains** that organise
the demo's gold tables, metric views, SQL functions, and Genie spaces into a
**business-aligned hierarchy** on the Databricks **Discover** page:

* **Domain:** SAP Purchasing (parent)
* **Subdomain 1:** Sourcing & Supplier Risk
* **Subdomain 2:** Purchase Order Analytics

Also **tags all 22 assets** (gold tables, metric views, SQL functions,
VALUE_DICTIONARY) and **publishes** the domains.

> **Pages** (business-term glossary) are handled separately in `09b_create_pages`.

> **Pre-requisites:** gold tables, semantic layer, and Genie spaces must exist
> (steps 1–6 in `00_run_all`).
>
> **Permission:** requires `MANAGE DISCOVERY` on the account.

In [0]:
%run ../00_config

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.tags import TagPolicy
import json, time

w = WorkspaceClient()
NS = CFG.namespace

# ---- Thin REST helper with proper headers ----
def _do(method, path, body=None, params=None):
    """Call the Databricks REST API via the SDK's authenticated client."""
    headers = {"Accept": "application/json", "Content-Type": "application/json"}
    if w.config.workspace_id:
        headers["X-Databricks-Workspace-Id"] = w.config.workspace_id
    return w.api_client.do(method, path, body=body, query=params, headers=headers)

print(f"Workspace: {w.config.host}")
print(f"Namespace: {NS}")

## 1) Create governed tags
Domains are built on **governed tags**. We create a parent tag and two sub-tags
following the `{parent}/{child}` naming convention that Databricks uses for
subdomains.

In [0]:
# ---- Governed-tag definitions ----
# Parent domain tag
PARENT_TAG = "SAP_Purchasing"
# Subdomain tags (must start with parent prefix + "/")
SUB_TAG_1 = f"{PARENT_TAG}/Sourcing_Supplier_Risk"
SUB_TAG_2 = f"{PARENT_TAG}/Purchase_Order_Analytics"


GOVERNED_TAGS_OK = True  # Track whether governed tags were created

def ensure_governed_tag(tag_key: str, description: str = ""):
    """Create a governed tag (tag policy) if it does not already exist.
    Uses the SDK TagPoliciesAPI (POST /api/2.1/tag-policies).
    Falls back gracefully if account tag-policy limit is reached.
    """
    global GOVERNED_TAGS_OK
    # First check if it already exists — if so, update its description.
    try:
        for p in w.tag_policies.list_tag_policies():
            if p.tag_key == tag_key:
                # Overwrite: update description to keep it current.
                try:
                    w.tag_policies.update_tag_policy(
                        id=p.id, policy=TagPolicy(tag_key=tag_key, description=description)
                    )
                    print(f"  ✓ Updated governed tag: {tag_key} (id={p.id})")
                except Exception:
                    print(f"  • Governed tag exists (update skipped): {tag_key} (id={p.id})")
                return True
    except Exception:
        pass
    # Try to create
    try:
        result = w.tag_policies.create_tag_policy(
            TagPolicy(tag_key=tag_key, description=description)
        )
        print(f"  ✓ Created governed tag: {tag_key} (id={result.id})")
        return True
    except Exception as e:
        if "ALREADY_EXISTS" in str(e) or "already exists" in str(e).lower():
            print(f"  • Governed tag already exists: {tag_key}")
            return True
        elif "maximum" in str(e).lower() and "tag policies" in str(e).lower():
            GOVERNED_TAGS_OK = False
            print(f"  ⚠ Account tag-policy limit reached — cannot create governed tag: {tag_key}")
            print(f"    The account has reached the max number of governed tags (tag policies).")
            print(f"    An account admin must delete unused tag policies to free capacity.")
            print(f"    Falling back to regular (non-governed) tags for asset tagging.")
            return False
        else:
            raise


tag1_ok = ensure_governed_tag(PARENT_TAG, "SAP Purchasing domain — parent tag for the Discover hierarchy.")
tag2_ok = ensure_governed_tag(SUB_TAG_1, "Sourcing & Supplier Risk subdomain.")
tag3_ok = ensure_governed_tag(SUB_TAG_2, "Purchase Order Analytics subdomain.")

if not (tag1_ok and tag2_ok and tag3_ok):
    print()
    print("=" * 72)
    print("⚠  ACCOUNT TAG-POLICY LIMIT REACHED")
    print("=" * 72)
    print("This account has hit the maximum number of governed tag policies.")
    print("Governed tags could NOT be created for this demo.")
    print()
    print("Impact:")
    print("  • Asset tagging (Step 3)   → proceeds with regular tags  ✓")
    print("  • Domain creation (Step 2) → SKIPPED (requires governed tags)")
    print("  • Pages creation (Step 4)  → SKIPPED (requires domains)")
    print("  • Domain publish (Step 5)  → SKIPPED (requires domains)")
    print()
    print("To resolve:")
    print("  An account admin must delete unused tag policies to free")
    print("  capacity, then re-run this notebook.")
    print("=" * 72)
else:
    print("\n✓ All governed tags ready.")

## 2) Create the domain and subdomains
The Domains REST API creates entries on the **Discover** page. A domain maps 1:1
to a governed tag; subdomains use the `{parentTag}/{name}` convention.

In [0]:
# ---- Discover the correct domains API path ----
# The Domains API lives at /api/2.0/domains (NOT under /unity-catalog/).
DOMAINS_API = None
for candidate in ["/api/2.0/domains", "/api/2.1/domains",
                  "/api/2.0/unity-catalog/domains", "/api/2.1/unity-catalog/domains"]:
    try:
        _do("GET", candidate)
        DOMAINS_API = candidate
        break
    except Exception as e:
        if "Not Found" not in str(e) and "No API found" not in str(e):
            DOMAINS_API = candidate
            break

if DOMAINS_API:
    print(f"Domains API found: {DOMAINS_API}")
else:
    print("\u26a0\ufe0f  Domains REST API not available on this workspace.")
    print("   The 'Domains and Discover Page' preview must be enabled.")
    print("   Governed tags + asset tagging will still proceed.")


def find_domain(tag_key: str):
    """Return the domain/subdomain ID for a given tag_key, or None."""
    if not DOMAINS_API:
        return None
    try:
        resp = _do("GET", DOMAINS_API)
        for d in resp.get("domains", []):
            if d.get("tag_key") == tag_key:
                return d.get("domain_id")
    except Exception:
        pass
    return None


def create_or_get_domain(tag_key: str, subtitle: str, description: str,
                         icon_name: str = "SHOPPING_CART", icon_color: str = "#1B5E20",
                         parent_domain_id: str = None):
    """Idempotent: create a domain/subdomain, or return existing ID."""
    if not GOVERNED_TAGS_OK:
        print(f"  • Skipped domain (governed tags unavailable): {tag_key}")
        return None
    if not DOMAINS_API:
        print(f"  • Skipped domain creation (API unavailable): {tag_key}")
        return None
    existing = find_domain(tag_key)
    body = {
        "tag_key": tag_key,
        "subtitle": subtitle,
        "description": description,
        "icon": {"name": icon_name, "color": icon_color},
    }
    if parent_domain_id:
        body["parent_domain_id"] = parent_domain_id
    if existing:
        # Overwrite: PATCH the existing domain to keep properties current.
        try:
            _do("PATCH", f"{DOMAINS_API}/{existing}", body=body)
            print(f"  ✓ Updated domain '{tag_key}' → {existing}")
        except Exception:
            print(f"  • Domain exists (update skipped): {tag_key} → {existing}")
        return existing
    try:
        resp = _do("POST", DOMAINS_API, body=body)
    except Exception as e:
        if "ALREADY_EXISTS" in str(e) or "already exists" in str(e).lower():
            print(f"  • Domain already exists (POST conflict): {tag_key}")
            return find_domain(tag_key)
        else:
            print(f"  ⚠ Domain creation failed for '{tag_key}': {e}")
            return None
    did = resp.get("domain_id")
    print(f"  ✓ Created domain '{tag_key}' → {did}")
    return did


# ---- Parent domain ----
domain_id = create_or_get_domain(
    tag_key=PARENT_TAG,
    subtitle="Sourcing risk, supplier management & PO analytics",
    description=(
        "SAP Purchasing data for an automotive OEM. Contains 5 gold tables, "
        "4 metric views, 7 certified SQL functions, 2 Genie spaces, and a "
        "Multi-Agent Supervisor — everything a purchasing analyst needs to "
        "answer sourcing-risk and supply-continuity questions in plain English."
    ),
    icon_name="SHOPPING_CART",
    icon_color="#1B5E20",
)
print(f"\nParent domain ID: {domain_id}")

# ---- Subdomain 1: Sourcing & Supplier Risk ----
sub1_id = create_or_get_domain(
    tag_key=SUB_TAG_1,
    subtitle="Multi-sourcing gaps, delivery risk, purchasing blocks & quotas",
    description=(
        "Answers questions about which materials lack multi-sourcing, which "
        "suppliers carry delivery risk, purchasing blocks with active sourcing, "
        "and quota concentration. Covers Q1–Q6, Q9–Q10 of the business question set."
    ),
    icon_name="WARNING",
    icon_color="#E65100",
    parent_domain_id=domain_id,
)

# ---- Subdomain 2: Purchase Order Analytics ----
sub2_id = create_or_get_domain(
    tag_key=SUB_TAG_2,
    subtitle="On-time delivery, price variance & orders under risk",
    description=(
        "Answers questions about on-time delivery rates, net-price variance "
        "across vendors, and purchase orders placed while a supplier had an "
        "active delivery-risk flag. Covers Q7, Q8, Q11."
    ),
    icon_name="TRUCK",
    icon_color="#1565C0",
    parent_domain_id=domain_id,
)

print(f"\nSubdomain 1 (Sourcing & Risk): {sub1_id}")
print(f"Subdomain 2 (PO Analytics):     {sub2_id}")

## 3) Assign assets to domains via governed tags
Apply the governed tag to each UC object (table, function, metric view) so it
appears under the correct domain/subdomain on the Discover page.

**Shared assets** (MATERIAL, SUPPLIER, VALUE_DICTIONARY) get the **parent** tag.
**Domain-specific assets** get the **subdomain** tag.

In [0]:
def set_tag(object_type: str, fq_name: str, tag_key: str):
    """Apply a governed tag to a UC object via SQL."""
    try:
        spark.sql(f"ALTER {object_type} {fq_name} SET TAGS ('{tag_key}' = '')")
        print(f"  ✓ Tagged {object_type} {fq_name.split('.')[-1]} → {tag_key}")
    except Exception as e:
        print(f"  • Tag {fq_name.split('.')[-1]}: {e}")


# ---- Shared assets → parent domain tag ----
print("\n--- Shared assets (parent domain) ---")
for tbl in ["MATERIAL", "SUPPLIER", "VALUE_DICTIONARY"]:
    set_tag("TABLE", CFG.fq(tbl), PARENT_TAG)

# ---- Subdomain 1: Sourcing & Supplier Risk ----
print("\n--- Subdomain 1: Sourcing & Supplier Risk ---")
for tbl in ["SOURCING", "SUPPLIER_RISK"]:
    set_tag("TABLE", CFG.fq(tbl), SUB_TAG_1)

# Metric views
for mv in ["mv_sourcing_coverage", "mv_supplier_risk"]:
    set_tag("TABLE", CFG.fq(mv), SUB_TAG_1)   # MVs are TABLE objects in UC

# SQL functions (the 7 we keep)
for fn in ["materials_without_multisourcing",
           "suppliers_with_delivery_risk",
           "materials_linked_to_risky_supplier",
           "single_sourced_risky",
           "blocked_suppliers_with_active_sourcing",
           "quota_concentration",
           # no-arg wrappers used by Genie
           "materials_without_multisourcing_default",
           "materials_linked_to_risky_supplier_today",
           "single_sourced_risky_today",
           "suppliers_with_delivery_risk_today"]:
    set_tag("FUNCTION", CFG.fq(fn), SUB_TAG_1)

# ---- Subdomain 2: Purchase Order Analytics ----
print("\n--- Subdomain 2: Purchase Order Analytics ---")
set_tag("TABLE", CFG.fq("PURCHASE_ORDER"), SUB_TAG_2)

for mv in ["mv_delivery_performance", "mv_price_analytics"]:
    set_tag("TABLE", CFG.fq(mv), SUB_TAG_2)

for fn in ["pos_under_active_risk",
           "otd_rate_per_supplier_last_12m"]:
    set_tag("FUNCTION", CFG.fq(fn), SUB_TAG_2)

print("\n✓ All assets tagged.")

## 4) Publish the domain and subdomains
Domains are created in **draft** state (visible only to curators). Publishing
makes them visible to all users on the Discover page.

In [0]:
def publish_domain(domain_id: str, tag_key: str):
    """Publish a domain/subdomain so it is visible to consumers."""
    if not domain_id or not DOMAINS_API:
        print(f"  • Skipped publish (domain unavailable): {tag_key}")
        return
    try:
        _do("POST", f"{DOMAINS_API}/{domain_id}/publish")
        print(f"  ✓ Published: {tag_key}")
    except Exception as e:
        if "already published" in str(e).lower():
            print(f"  • Already published: {tag_key}")
        else:
            print(f"  • Publish {tag_key}: {e}")


publish_domain(domain_id, PARENT_TAG)
publish_domain(sub1_id, SUB_TAG_1)
publish_domain(sub2_id, SUB_TAG_2)

print(f"\n✓ Domain publish step complete.")

## 5) Persist domain IDs for teardown

In [0]:
from pyspark.sql import Row

spark.createDataFrame([
    Row(domain_key=PARENT_TAG, domain_id=str(domain_id), domain_type="domain"),
    Row(domain_key=SUB_TAG_1, domain_id=str(sub1_id), domain_type="subdomain"),
    Row(domain_key=SUB_TAG_2, domain_id=str(sub2_id), domain_type="subdomain"),
]).write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
 .saveAsTable(CFG.fq("_DOMAINS"))

print("Domain IDs saved to _DOMAINS:")
display(spark.table(CFG.fq("_DOMAINS")))

domains_ok = DOMAINS_API is not None
tags_ok = GOVERNED_TAGS_OK
result_parts = []
if tags_ok:
    result_parts.append("governed-tags=OK")
else:
    result_parts.append("governed-tags=SKIPPED(account-limit)")
result_parts.append("asset-tagging=OK")
if domains_ok and domain_id:
    result_parts.append(f"domains={domain_id}")
elif not tags_ok:
    result_parts.append("domains=SKIPPED(governed-tags-unavailable)")
else:
    result_parts.append("domains=SKIPPED(preview-not-enabled)")

print(f"\nOpen Discover: {w.config.host}/discover")
dbutils.notebook.exit(f"09a_create_domains: SUCCESS — {', '.join(result_parts)}")